In [1]:
import pandas as pd
import numpy as np
df = pd.read_csv("archive (1).zip")
df.head(5)

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,ERROR,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,UNKNOWN,UNKNOWN,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11


In [2]:
print("Exploratory analysis")
print("----------------DATA TYPES--------")
print(df.info())
print("---------------------MISSING VALUES----------")
print(df.isnull().sum())
print("-------------------DUPLICATES----------")
print(df.duplicated().sum())
print(f"No. of rows and columns {df.shape} ")

Exploratory analysis
----------------DATA TYPES--------
<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   Transaction ID    10000 non-null  str  
 1   Item              9667 non-null   str  
 2   Quantity          9862 non-null   str  
 3   Price Per Unit    9821 non-null   str  
 4   Total Spent       9827 non-null   str  
 5   Payment Method    7421 non-null   str  
 6   Location          6735 non-null   str  
 7   Transaction Date  9841 non-null   str  
dtypes: str(8)
memory usage: 625.1 KB
None
---------------------MISSING VALUES----------
Transaction ID         0
Item                 333
Quantity             138
Price Per Unit       179
Total Spent          173
Payment Method      2579
Location            3265
Transaction Date     159
dtype: int64
-------------------DUPLICATES----------
0
No. of rows and columns (10000, 8) 


In [3]:
null_count_before = df.isnull().sum().sum()
df = df.replace(["UNKNOWN", "ERROR"], np.nan)
null_count_after = df.isnull().sum().sum()
print(f'''No of "ERRORS" and "UNKNOWN" found in data was {null_count_before - null_count_after}''')
print(df.isnull().sum())
df.head(5)

No of "ERRORS" and "UNKNOWN" found in data was -3256
Transaction ID         0
Item                 969
Quantity             479
Price Per Unit       533
Total Spent          502
Payment Method      3178
Location            3961
Transaction Date     460
dtype: int64


,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,NaN,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,NaN,NaN,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11


In [4]:
df["Quantity"] = pd.to_numeric(df["Quantity"], errors = "coerce")
df["Price Per Unit"] = pd.to_numeric(df["Price Per Unit"], errors = "coerce")
df["Total Spent"] = pd.to_numeric(df["Total Spent"], errors="coerce")
df["Transaction Date"] = pd.to_datetime(df["Transaction Date"], errors="coerce")
print(df.info())

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   Transaction ID    10000 non-null  str           
 1   Item              9031 non-null   str           
 2   Quantity          9521 non-null   float64       
 3   Price Per Unit    9467 non-null   float64       
 4   Total Spent       9498 non-null   float64       
 5   Payment Method    6822 non-null   str           
 6   Location          6039 non-null   str           
 7   Transaction Date  9540 non-null   datetime64[us]
dtypes: datetime64[us](1), float64(3), str(4)
memory usage: 625.1 KB
None


In [5]:
df["Item"] = df["Item"].fillna("Unknown Item")
df["Payment Method"] = df["Payment Method"].fillna("Unknown")
df["Location"] = df["Location"].fillna("Unknown")
no_of_rows_before = df.shape[0]
df = df.dropna(subset="Transaction Date").reset_index(drop = True)
no_of_rows_after = df.shape[0]
print("\n Total droped rows:", no_of_rows_before - no_of_rows_after)
print(df.isnull().sum())


 Total droped rows: 460
Transaction ID        0
Item                  0
Quantity            454
Price Per Unit      506
Total Spent         476
Payment Method        0
Location              0
Transaction Date      0
dtype: int64


In [6]:
complete = df.dropna(subset = ["Price Per Unit", "Total Spent", "Quantity"] )

expected = (complete["Quantity"] * complete["Price Per Unit"]).round(2)
actual = complete["Total Spent"].round(2)

mismatch = complete[expected != actual]
if len(mismatch) > 0:
    print("----------SAMPLE MISMATCHES------")
    print(mismatch[["Total Spent", "Quantity", "Price Per Unit"]].head(5))

has_both = df["Quantity"].notna() & df["Price Per Unit"].notna()
df.loc[has_both, "Total Spent"] = (df.loc[has_both, "Quantity"] * df.loc[has_both, "Price Per Unit"]).round(2)


median_spent = df["Total Spent"].median()
df["Total Spent"] = df["Total Spent"].fillna(median_spent)

print("Null Total spent values has been filled with", median_spent)

print(df.isnull().sum())

Null Total spent values has been filled with 8.0
Transaction ID        0
Item                  0
Quantity            454
Price Per Unit      506
Total Spent           0
Payment Method        0
Location              0
Transaction Date      0
dtype: int64


In [7]:
print("-------------------REPORT-------------")
print(f'''Initial Null Values: {null_count_before}\n
Initial number of rows: {no_of_rows_before}\n
Before cleaning (10000, 8)
Dataset contained {null_count_after - null_count_before} "UNKNOWNS & ERRORS"\n
UNKNOWNS AND ERRORS were cleaned out bring total null vales to {null_count_after}\n
Data types for Quantity, Total Spent, Price Per Unit, Transaction date were updated\n
Misiing values in "Item" column were filled with "Unknown Items" (969) missing values\n
3178 missing values in "Payment method" column was filled with "Unknown"\n
3961 missing values in "Location" column was filled with "Unknown"\n
Rows with missing values for Transaction Date was dropped (460)\n
missing values in Total Spent column was filled with its median\n
Final null values: {df.isnull().sum().sum()}\n
Final number of rows: {no_of_rows_after}\n
After cleaning {df.shape}
Final Null Values: {df.isnull().sum()}
''')


-------------------REPORT-------------
Initial Null Values: 6826

Initial number of rows: 10000

Before cleaning (10000, 8)
Dataset contained 3256 "UNKNOWNS & ERRORS"

UNKNOWNS AND ERRORS were cleaned out bring total null vales to 10082

Data types for Quantity, Total Spent, Price Per Unit, Transaction date were updated

Misiing values in "Item" column were filled with "Unknown Items" (969) missing values

3178 missing values in "Payment method" column was filled with "Unknown"

3961 missing values in "Location" column was filled with "Unknown"

Rows with missing values for Transaction Date was dropped (460)

missing values in Total Spent column was filled with its median

Final null values: 960

Final number of rows: 9540

After cleaning (9540, 8)
Final Null Values: Transaction ID        0
Item                  0
Quantity            454
Price Per Unit      506
Total Spent           0
Payment Method        0
Location              0
Transaction Date      0
dtype: int64

